# 血液抹片細胞自動辨識與計數系統
> 影像處理 114-2 ｜ 期末專案

## 系統說明
本系統使用傳統影像處理技術（不使用深度學習）自動偵測血液抹片中的三種細胞：
- **WBC**（白血球）：灰階閾值 AND 紫色 HSV 遮罩 → 形態學操作 → 面積篩選（> 9500 px²）
- **RBC**（紅血球）：Watershed + Distance Transform 分離重疊細胞，各分割區域計為一個 RBC
- **PLT**（血小板）：紫色 HSV 遮罩 → 面積篩選（400–3000 px²），排除 WBC + RBC 區域

## 評估結果（Test 集，126 張）— **Watershed 版**
| 細胞 | GT | 預測 | 誤差 | 狀態 |
|------|----|------|------|------|
| WBC  | 133| —    | —    | 待 Colab 重測 |
| RBC  |1699| —    | —    | 待 Colab 重測 |
| PLT  | 49 | —    | —    | 待 Colab 重測 |

> 執行 Cell 9 後更新此表格。舊版（Hough 偵測）結果：WBC +6.8%, RBC +8.8%, PLT +16.3%

## Cell 1 — 安裝依賴套件

In [ ]:
# 安裝 Streamlit 及 Colab 展示工具
!pip install -q streamlit pyngrok opencv-python-headless
print('安裝完成')

安裝完成


## Cell 2 — 匯入套件

In [ ]:
import cv2
import numpy as np
import glob
import os
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('OpenCV 版本:', cv2.__version__)
print('NumPy 版本:', np.__version__)

OpenCV 版本: 4.13.0
NumPy 版本: 2.0.2


## Cell 3 — 資料集路徑設定

如果在 Google Colab 執行，請先掛載 Drive 或 clone 資料集：
```bash
# 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 或 clone 資料集
!git clone https://github.com/lugan113/TXL-PBC_Dataset.git
```

In [ ]:
import os, glob, subprocess

REPO_URL = 'https://github.com/lugan113/TXL-PBC_Dataset.git'

def _find_dataset():
    cwd = os.getcwd()
    candidates = [
        os.path.join(cwd, 'TXL-PBC_Dataset', 'TXL-PBC'),
        os.path.join(cwd, 'TXL-PBC_Dataset'),
        '/content/TXL-PBC_Dataset/TXL-PBC',
        '/content/TXL-PBC_Dataset',
        '/content/drive/MyDrive/TXL-PBC_Dataset/TXL-PBC',
        '/content/drive/MyDrive/TXL-PBC_Dataset',
        'TXL-PBC_Dataset/TXL-PBC',
        'TXL-PBC_Dataset',
    ]
    for p in candidates:
        if os.path.isdir(os.path.join(p, 'images')):
            return p
    return None

DATASET_ROOT = _find_dataset()

# 找不到時，在 Colab 環境自動 clone
if DATASET_ROOT is None and os.path.isdir('/content'):
    print(f'未找到資料集，自動從 GitHub clone...')
    ret = subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL],
        cwd='/content', capture_output=True, text=True
    )
    if ret.returncode == 0:
        print('Clone 完成，重新搜尋資料集...')
        DATASET_ROOT = _find_dataset()
    else:
        print('Clone 失敗：', ret.stderr[:300])

if DATASET_ROOT is None:
    raise FileNotFoundError(
        '找不到 TXL-PBC 資料集！\n'
        f'工作目錄：{os.getcwd()}\n'
        '請手動執行：!git clone https://github.com/lugan113/TXL-PBC_Dataset.git\n'
        '然後重新執行此 cell。'
    )

print(f'資料集路徑：{DATASET_ROOT}')

SPLIT     = 'test'
IMG_DIR   = f'{DATASET_ROOT}/images/{SPLIT}'
LABEL_DIR = f'{DATASET_ROOT}/labels/{SPLIT}'
img_files = sorted(glob.glob(f'{IMG_DIR}/*.png'))

if not img_files:
    raise FileNotFoundError(
        f'在 {IMG_DIR} 找不到任何 .png 影像！\n'
        '請確認資料集完整，或更換 SPLIT 值（test/val/train）。'
    )

print(f'找到 {len(img_files)} 張 {SPLIT} 影像')

CLASS_NAMES  = {0: 'WBC', 1: 'RBC', 2: 'PLT'}
CLASS_COLORS = {'WBC': (0, 200, 0), 'RBC': (0, 0, 220), 'PLT': (220, 0, 0)}

資料集路徑：/content/TXL-PBC_Dataset/TXL-PBC
找到 126 張 test 影像


## Cell 4 — 偵測參數設定

所有可調整的參數集中於此，方便在 Streamlit App 中對應 slider 控制。

In [ ]:
# ── WBC 偵測參數 ──────────────────────────────────────────────────
WBC_GRAY_THRESH = 130
WBC_CLOSE_K     = 23
WBC_OPEN_K      = 9
WBC_MIN_AREA    = 9500   # px²
WBC_PAD         = 65

# ── RBC 偵測參數（Hough Circle Transform）────────────────────────
RBC_MIN_DIST = 38
RBC_PARAM1   = 50
RBC_PARAM2   = 12
RBC_MIN_R    = 28
RBC_MAX_R    = 78

# ── PLT 偵測參數（HSV 紫色遮罩）──────────────────────────────────
PLT_H_LO     = 112
PLT_H_HI     = 162
PLT_S_MIN    = 38
PLT_V_HI     = 210
PLT_MIN_AREA = 400
PLT_MAX_AREA = 3000

# ── Watershed RBC 分離參數 ────────────────────────────────────────
WS_CLOSE_K     = 18    # closing kernel：填滿 RBC 雙凹中心
WS_DIST_THRESH = 0.40  # distance transform 峰值閾值（越高→保守，越低→偵測更多）

print('參數設定完成')

## Cell 5 — 輔助函式（GT 解析、評估）

In [ ]:
IOU_THRESH = 0.5


def load_gt(label_path, img_w=575, img_h=575):
    """解析 YOLO 格式標籤，回傳各類別 bounding box 清單。"""
    gt = {'WBC': [], 'RBC': [], 'PLT': []}
    if not os.path.exists(label_path):
        return gt
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = round((cx - bw/2) * img_w)
            y1 = round((cy - bh/2) * img_h)
            x2 = round((cx + bw/2) * img_w)
            y2 = round((cy + bh/2) * img_h)
            name = CLASS_NAMES.get(cls)
            if name:
                gt[name].append([x1, y1, x2, y2])
    return gt


def iou(boxA, boxB):
    """計算兩個 [x1,y1,x2,y2] 框的 Intersection over Union。"""
    x_left   = max(boxA[0], boxB[0])
    y_top    = max(boxA[1], boxB[1])
    x_right  = min(boxA[2], boxB[2])
    y_bottom = min(boxA[3], boxB[3])
    inter = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    if inter == 0:
        return 0.0
    area_a = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    area_b = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / (area_a + area_b - inter)


def match_detections(gt_boxes, pred_boxes, iou_thresh=IOU_THRESH):
    """Greedy 一對一 IoU 配對，回傳 (TP, FP, FN)。"""
    matched_gt   = set()
    matched_pred = set()
    pairs = []
    for i, gb in enumerate(gt_boxes):
        for j, pb in enumerate(pred_boxes):
            score = iou(gb, pb)
            if score >= iou_thresh:
                pairs.append((score, i, j))
    pairs.sort(reverse=True)
    for _, i, j in pairs:
        if i not in matched_gt and j not in matched_pred:
            matched_gt.add(i)
            matched_pred.add(j)
    tp = len(matched_gt)
    fp = len(pred_boxes) - len(matched_pred)
    fn = len(gt_boxes)   - len(matched_gt)
    return tp, fp, fn


def draw_gt(img, gt_boxes):
    """在影像上繪製 Ground Truth bounding boxes（細線）。"""
    GT_COLORS = {'WBC': (0, 220, 0), 'RBC': (220, 180, 0), 'PLT': (200, 0, 200)}
    out = img.copy()
    for name, boxes in gt_boxes.items():
        color = GT_COLORS[name]
        for box in boxes:
            x1, y1, x2, y2 = box
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 1)
            cv2.putText(out, name, (x1, max(0, y1-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
    return out


print('輔助函式載入完成（IoU 閾值：', IOU_THRESH, '）')

## Cell 6 — 影像前處理 Pipeline 函式

依序回傳各階段影像：原始 → 灰階 → Gaussian Blur → Adaptive Thresholding → Morphological Closing

In [ ]:
def preprocess_pipeline(img_bgr, wbc_gray_thresh=WBC_GRAY_THRESH):
    """
    標準前處理 Pipeline（可視化用）。

    回傳:
        steps: dict，包含各階段影像
            'original'      : 原始 BGR 影像
            'grayscale'     : 灰階影像
            'gaussian_blur' : Gaussian Blur 後影像
            'adaptive_thr'  : Adaptive Thresholding 結果（反轉：細胞=白）
            'morph_close'   : Morphological Closing 後（用於 WBC 核心）
    """
    # Step 1: 灰階轉換
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Step 2: Gaussian Blur（去除雜訊）
    blurred = cv2.GaussianBlur(gray, (11, 11), 2)

    # Step 3: Adaptive Thresholding（局部自適應閾值）
    # THRESH_BINARY_INV：細胞（較暗）→ 白色
    adaptive = cv2.adaptiveThreshold(
        blurred, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=51, C=8
    )

    # Step 4: Morphological Closing（填補細胞核內部空洞）
    kernel = np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8)
    morph_close = cv2.morphologyEx(adaptive, cv2.MORPH_CLOSE, kernel)

    # 同時準備 WBC 全域閾值版本（用於偵測 WBC 細胞核）
    _, dark_global = cv2.threshold(blurred, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    morph_close_global = cv2.morphologyEx(
        dark_global, cv2.MORPH_CLOSE, np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8)
    )

    return {
        'original':      img_bgr,
        'grayscale':     gray,
        'gaussian_blur': blurred,
        'adaptive_thr':  adaptive,
        'morph_close':   morph_close_global,  # 形態學操作（全域閾值版，更清楚展示 WBC）
    }


print('Pipeline 函式載入完成')

Pipeline 函式載入完成


## Cell 7 — 細胞偵測函式

整合 WBC / RBC / PLT 偵測，並回傳帶框線的結果影像。

### RBC 偵測策略：Watershed + Distance Transform

相鄰或重疊的 RBC 會形成連通的細胞團，直接用輪廓面積會把整群算成一個細胞。
Watershed 透過以下步驟切分：

1. **Adaptive Threshold → Morphological Closing** — 填滿 RBC 雙凹中心，得到 solid disc
2. **Distance Transform** — 計算每個前景像素到背景的最短距離；每個細胞中心出現峰值
3. **峰值篩選 → 種子標記** — 超過 `WS_DIST_THRESH × max` 的像素為確定前景（每個連通峰值 = 一個種子）
4. **Watershed** — 從種子向外 flood fill，直到遇到相鄰細胞的邊界（標記為 -1）
5. **每個分割區域 → minEnclosingCircle** — 取得 (cx, cy, r) 供計數與繪圖

結果影像中，黃色細線為 Watershed 分割邊界，清楚顯示重疊細胞被成功切分。

In [ ]:
def _hough_seeded_watershed(img_bgr, rbc_centers, h, w):
    """
    Hough-seeded Watershed：直接用 Hough 圓繪製細胞遮罩為種子區域，
    確保相鄰 RBC 之間一定出現 Watershed 分割邊界（黃線）。
    回傳 ws_markers（-1 = 邊界像素）。
    """
    cell_mask = np.zeros((h, w), np.uint8)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(cell_mask, (cx, cy), r, 255, -1)
    cell_union = cv2.dilate(cell_mask, np.ones((7, 7), np.uint8), iterations=1)

    markers = np.zeros((h, w), np.int32)
    markers[cell_union == 0] = 1   # 確定背景 → label 1
    for i, (cx, cy, r) in enumerate(rbc_centers):
        seed_r = max(3, r // 4)
        cx_c = int(np.clip(cx, 0, w - 1))
        cy_c = int(np.clip(cy, 0, h - 1))
        cv2.circle(markers, (cx_c, cy_c), seed_r, i + 2, -1)

    markers = cv2.watershed(img_bgr.copy(), markers.copy())
    return markers


def _circularity(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    return (4 * np.pi * area / (peri ** 2)) if peri > 0 else 0.0


def detect_cells(img_bgr,
                 wbc_gray_thresh=WBC_GRAY_THRESH,
                 rbc_param2=RBC_PARAM2,
                 plt_min_area=PLT_MIN_AREA):
    """
    偵測三種血液細胞。

    回傳：
        boxes   : {'WBC': [[x1,y1,x2,y2],...], 'RBC': [...], 'PLT': [...]}
        result  : 帶標注的結果影像（BGR）
        details : 原始輪廓/圓心等中間資料
    """
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # ── Step 1: WBC ────────────────────────────────────────────────────
    g_blur = cv2.GaussianBlur(gray, (11, 11), 2)
    _, dark = cv2.threshold(g_blur, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    purple  = cv2.inRange(hsv, (100, 25, 20), (168, 255, 200))
    dark    = cv2.bitwise_and(dark, purple)
    dark    = cv2.morphologyEx(dark, cv2.MORPH_CLOSE,
                               np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8))
    dark    = cv2.morphologyEx(dark, cv2.MORPH_OPEN,
                               np.ones((WBC_OPEN_K,  WBC_OPEN_K),  np.uint8))
    all_cnts, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    wbc_cnts = [c for c in all_cnts if cv2.contourArea(c) >= WBC_MIN_AREA]

    wbc_excl = np.zeros((h, w), np.uint8)
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(wbc_excl,
                      (max(0, x - WBC_PAD),      max(0, y - WBC_PAD)),
                      (min(w, x + bw + WBC_PAD), min(h, y + bh + WBC_PAD)),
                      255, -1)

    # ── Step 2a: RBC 計數 — Hough Circle Transform ─────────────────────
    blur5   = cv2.GaussianBlur(gray, (5, 5), 1)
    circles = cv2.HoughCircles(
        blur5, cv2.HOUGH_GRADIENT, dp=1,
        minDist=RBC_MIN_DIST, param1=RBC_PARAM1, param2=rbc_param2,
        minRadius=RBC_MIN_R,  maxRadius=RBC_MAX_R
    )
    rbc_centers = []
    if circles is not None:
        for (cx, cy, r) in np.round(circles[0]).astype(int):
            cc = int(np.clip(cx, 0, w-1)); rc = int(np.clip(cy, 0, h-1))
            if wbc_excl[rc, cc] == 0:
                rbc_centers.append((int(cx), int(cy), int(r)))

    # ── Step 2b: Hough-seeded Watershed — 視覺化相鄰 RBC 分割邊界 ─────
    ws_markers = _hough_seeded_watershed(img_bgr, rbc_centers, h, w)

    # ── Step 3: PLT ────────────────────────────────────────────────────
    rbc_excl = np.zeros((h, w), np.uint8)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(rbc_excl, (cx, cy), r + 10, 255, -1)

    all_excl  = cv2.bitwise_or(wbc_excl, rbc_excl)
    purp_mask = cv2.inRange(hsv, (PLT_H_LO, PLT_S_MIN, 30), (PLT_H_HI, 255, PLT_V_HI))
    purp_mask = cv2.bitwise_and(purp_mask, cv2.bitwise_not(all_excl))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_OPEN,  np.ones((2, 2), np.uint8))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_CLOSE, np.ones((4, 4), np.uint8))
    plt_all, _ = cv2.findContours(purp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plt_dets   = [c for c in plt_all if plt_min_area <= cv2.contourArea(c) <= PLT_MAX_AREA]

    # ── 收集 Bounding Boxes ────────────────────────────────────────────
    wbc_boxes = []
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        wbc_boxes.append([x, y, x + bw, y + bh])

    rbc_boxes = []
    for (cx, cy, r) in rbc_centers:
        rbc_boxes.append([max(0, cx - r), max(0, cy - r),
                          min(w, cx + r), min(h, cy + r)])

    plt_boxes = []
    for cnt in plt_dets:
        x, y, bw, bh = cv2.boundingRect(cnt)
        plt_boxes.append([x, y, x + bw, y + bh])

    # ── 繪製偵測結果 ───────────────────────────────────────────────────
    result = img_bgr.copy()
    boundary = (ws_markers == -1).astype(np.uint8)
    boundary = cv2.dilate(boundary, np.ones((2, 2), np.uint8), iterations=1)
    result[boundary > 0] = [0, 220, 220]  # 黃色 Watershed 邊界

    for box in wbc_boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(result, (x1, y1), (x2, y2), CLASS_COLORS['WBC'], 2)
        cv2.putText(result, 'WBC', (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, CLASS_COLORS['WBC'], 1)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(result, (cx, cy), r, CLASS_COLORS['RBC'], 1)
    for box in plt_boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(result, (x1, y1), (x2, y2), CLASS_COLORS['PLT'], 1)

    boxes   = {'WBC': wbc_boxes, 'RBC': rbc_boxes, 'PLT': plt_boxes}
    details = {'wbc_cnts': wbc_cnts, 'rbc_centers': rbc_centers,
               'plt_dets': plt_dets, 'ws_markers': ws_markers}
    return boxes, result, details


print('偵測函式載入完成（回傳 bounding boxes，支援 IoU 評估）')

## Cell 8 — 快速測試（單張影像視覺化）

對第一張測試影像執行完整 pipeline，並用 matplotlib 顯示各階段結果。

In [ ]:
# 選擇測試影像（可修改索引）
TEST_IDX = 0
img_path = img_files[TEST_IDX]
stem = Path(img_path).stem

img = cv2.imread(img_path)
gt  = load_gt(f'{LABEL_DIR}/{stem}.txt')

# 執行前處理 pipeline
steps = preprocess_pipeline(img)

# 執行細胞偵測
boxes, result, details = detect_cells(img)

# ── 視覺化：前處理各階段 ─────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle(f'前處理 Pipeline — {stem[:30]}', fontsize=12)

titles = ['① 原始影像', '② 灰階', '③ Gaussian Blur', '④ Morph. Closing (WBC)']
images = [
    cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
    steps['grayscale'],
    steps['gaussian_blur'],
    steps['morph_close'],
]
cmaps = ['viridis', 'gray', 'gray', 'gray']
for ax, title, image, cmap in zip(axes, titles, images, cmaps):
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig('pipeline_demo.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 視覺化：Ground Truth vs 預測結果 ─────────────────────────────
fig2, (ax_gt, ax_pred) = plt.subplots(1, 2, figsize=(12, 5))
fig2.suptitle(f'Ground Truth vs 預測結果 — {stem[:30]}', fontsize=12)

ax_gt.imshow(cv2.cvtColor(draw_gt(img, gt), cv2.COLOR_BGR2RGB))
ax_gt.set_title('Ground Truth', fontsize=11)
ax_gt.axis('off')

ax_pred.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
ax_pred.set_title('預測結果（黃線 = Watershed RBC 分割邊界）', fontsize=11)
ax_pred.axis('off')

plt.tight_layout()
plt.savefig('gt_vs_pred.png', dpi=120, bbox_inches='tight')
plt.show()

# ── IoU 指標計算 ─────────────────────────────────────────────────
print(f'\n{"─"*60}')
print(f'{"Cell":6s} {"GT":>5s} {"Pred":>5s} {"TP":>5s} {"FP":>5s} {"FN":>5s} {"Precision":>10s} {"Recall":>8s}')
print(f'{"─"*60}')
for name in ['WBC', 'RBC', 'PLT']:
    gt_boxes   = gt[name]
    pred_boxes = boxes[name]
    tp, fp, fn = match_detections(gt_boxes, pred_boxes)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    print(f'{name:6s} {len(gt_boxes):>5d} {len(pred_boxes):>5d} '
          f'{tp:>5d} {fp:>5d} {fn:>5d} {prec:>10.1%} {rec:>8.1%}')
print(f'{"─"*60}')
print(f'IoU 閾值：{IOU_THRESH}')

## Cell 9 — 全資料集評估

In [ ]:
from tqdm.notebook import tqdm

def evaluate_dataset(split='test', iou_thresh=IOU_THRESH):
    img_dir   = f'{DATASET_ROOT}/images/{split}'
    label_dir = f'{DATASET_ROOT}/labels/{split}'
    files     = sorted(glob.glob(f'{img_dir}/*.png'))

    totals = {name: {'tp': 0, 'fp': 0, 'fn': 0} for name in ['WBC', 'RBC', 'PLT']}

    for img_path in tqdm(files, desc=f'評估 {split}', unit='張'):
        stem = Path(img_path).stem
        img  = cv2.imread(img_path)
        gt   = load_gt(f'{label_dir}/{stem}.txt')
        pred_boxes, _, _ = detect_cells(img)
        for name in ['WBC', 'RBC', 'PLT']:
            tp, fp, fn = match_detections(gt[name], pred_boxes[name], iou_thresh)
            totals[name]['tp'] += tp
            totals[name]['fp'] += fp
            totals[name]['fn'] += fn

    print(f'\n資料集評估結果 — Split: {split}  ({len(files)} 張影像)  IoU≥{iou_thresh}')
    print(f'{"═"*64}')
    print(f'{"細胞":6s} {"TP":>6s} {"FP":>6s} {"FN":>6s} {"Precision":>10s} {"Recall":>8s}')
    print(f'{"─"*64}')
    for name in ['WBC', 'RBC', 'PLT']:
        tp = totals[name]['tp']
        fp = totals[name]['fp']
        fn = totals[name]['fn']
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        print(f'{name:6s} {tp:>6d} {fp:>6d} {fn:>6d} {prec:>10.1%} {rec:>8.1%}')
    print(f'{"═"*64}')

evaluate_dataset('test')

## Cell 10 — 撰寫 Streamlit App

使用 `%%writefile` 將 app 程式碼存至 `app.py`，再透過 Streamlit 啟動互動式介面。

**App 功能：**
- 圖片切換功能（slider 選擇影像）
- 3 個可調參數（WBC 閾值、RBC 靈敏度、PLT 最小面積）
- 各階段前處理結果顯示
- Ground Truth 對照
- 下載偵測結果按鈕

In [ ]:
%%writefile app.py
"""
血液抹片細胞自動辨識與計數系統 — Streamlit App
影像處理 114-2 期末專案
"""

import streamlit as st
import cv2
import numpy as np
import glob
import os
from pathlib import Path

# ─────────────────────── 資料集路徑自動偵測 ──────────────────────────
def _find_dataset():
    cwd = os.getcwd()
    candidates = [
        os.path.join(cwd, 'TXL-PBC_Dataset', 'TXL-PBC'),
        os.path.join(cwd, 'TXL-PBC_Dataset'),
        '/content/TXL-PBC_Dataset/TXL-PBC',
        '/content/TXL-PBC_Dataset',
        '/content/drive/MyDrive/TXL-PBC_Dataset/TXL-PBC',
        '/content/drive/MyDrive/TXL-PBC_Dataset',
    ]
    for p in candidates:
        if os.path.isdir(os.path.join(p, 'images')):
            return p
    return None

DATASET_ROOT = _find_dataset()
DEMO_COUNT   = 5
IOU_THRESH   = 0.5

# ─────────────────────── 常數 ────────────────────────────────────────
CLASS_NAMES  = {0: 'WBC', 1: 'RBC', 2: 'PLT'}
CLASS_COLORS = {'WBC': (0, 200, 0), 'RBC': (0, 0, 220), 'PLT': (220, 0, 0)}
GT_COLORS    = {'WBC': (0, 220, 0), 'RBC': (220, 180, 0), 'PLT': (200, 0, 200)}

WBC_CLOSE_K  = 23;   WBC_OPEN_K   = 9
WBC_MIN_AREA = 9500; WBC_PAD      = 65
RBC_MIN_DIST = 38;   RBC_PARAM1   = 50
RBC_MIN_R    = 28;   RBC_MAX_R    = 78
PLT_H_LO     = 112;  PLT_H_HI    = 162
PLT_S_MIN    = 38;   PLT_V_HI    = 210;  PLT_MAX_AREA = 3000


# ─────────────────────── 工具函式 ────────────────────────────────────
@st.cache_data
def load_image_list(split):
    if DATASET_ROOT is None:
        return []
    return sorted(glob.glob(f'{DATASET_ROOT}/images/{split}/*.png'))[:DEMO_COUNT]


def load_gt(label_path, img_w=575, img_h=575):
    gt = {'WBC': [], 'RBC': [], 'PLT': []}
    if not os.path.exists(label_path):
        return gt
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = round((cx - bw/2) * img_w);  x2 = round((cx + bw/2) * img_w)
            y1 = round((cy - bh/2) * img_h);  y2 = round((cy + bh/2) * img_h)
            name = CLASS_NAMES.get(cls)
            if name:
                gt[name].append([x1, y1, x2, y2])
    return gt


def iou(boxA, boxB):
    x_left   = max(boxA[0], boxB[0]); y_top    = max(boxA[1], boxB[1])
    x_right  = min(boxA[2], boxB[2]); y_bottom = min(boxA[3], boxB[3])
    inter = max(0, x_right - x_left) * max(0, y_bottom - y_top)
    if inter == 0:
        return 0.0
    area_a = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    area_b = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    return inter / (area_a + area_b - inter)


def match_detections(gt_boxes, pred_boxes, iou_thresh=IOU_THRESH):
    matched_gt = set(); matched_pred = set(); pairs = []
    for i, gb in enumerate(gt_boxes):
        for j, pb in enumerate(pred_boxes):
            score = iou(gb, pb)
            if score >= iou_thresh:
                pairs.append((score, i, j))
    pairs.sort(reverse=True)
    for _, i, j in pairs:
        if i not in matched_gt and j not in matched_pred:
            matched_gt.add(i); matched_pred.add(j)
    return len(matched_gt), len(pred_boxes) - len(matched_pred), len(gt_boxes) - len(matched_gt)


def bgr2rgb(img): return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
def gray2rgb(img): return cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)


# ─────────────────────── 前處理 Pipeline ─────────────────────────────
def preprocess_pipeline(img_bgr, wbc_gray_thresh):
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (11, 11), 2)
    adaptive = cv2.adaptiveThreshold(blurred, 255,
                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 51, 8)
    _, dark = cv2.threshold(blurred, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    morph_close = cv2.morphologyEx(dark, cv2.MORPH_CLOSE,
                                   np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8))
    return {'grayscale': gray, 'gaussian_blur': blurred,
            'adaptive_thr': adaptive, 'morph_close': morph_close}


# ─────────────────────── Hough-seeded Watershed ───────────────────────
def _hough_seeded_watershed(img_bgr, rbc_centers, h, w):
    cell_mask = np.zeros((h, w), np.uint8)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(cell_mask, (cx, cy), r, 255, -1)
    cell_union = cv2.dilate(cell_mask, np.ones((7, 7), np.uint8), iterations=1)
    markers = np.zeros((h, w), np.int32)
    markers[cell_union == 0] = 1
    for i, (cx, cy, r) in enumerate(rbc_centers):
        seed_r = max(3, r // 4)
        cx_c = int(np.clip(cx, 0, w - 1)); cy_c = int(np.clip(cy, 0, h - 1))
        cv2.circle(markers, (cx_c, cy_c), seed_r, i + 2, -1)
    markers = cv2.watershed(img_bgr.copy(), markers.copy())
    return markers


# ─────────────────────── 細胞偵測 ────────────────────────────────────
@st.cache_data
def detect_cells_cached(img_bytes, wbc_gray_thresh, rbc_param2, plt_min_area):
    img_arr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
    return _detect(img, wbc_gray_thresh, rbc_param2, plt_min_area)


def _detect(img_bgr, wbc_gray_thresh, rbc_param2, plt_min_area):
    h, w = img_bgr.shape[:2]
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # Step 1: WBC
    g_blur = cv2.GaussianBlur(gray, (11, 11), 2)
    _, dark = cv2.threshold(g_blur, wbc_gray_thresh, 255, cv2.THRESH_BINARY_INV)
    purple  = cv2.inRange(hsv, (100, 25, 20), (168, 255, 200))
    dark    = cv2.bitwise_and(dark, purple)
    dark    = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, np.ones((WBC_CLOSE_K, WBC_CLOSE_K), np.uint8))
    dark    = cv2.morphologyEx(dark, cv2.MORPH_OPEN,  np.ones((WBC_OPEN_K,  WBC_OPEN_K),  np.uint8))
    all_cnts, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    wbc_cnts = [c for c in all_cnts if cv2.contourArea(c) >= WBC_MIN_AREA]
    wbc_excl = np.zeros((h, w), np.uint8)
    for cnt in wbc_cnts:
        x, y, bw, bh = cv2.boundingRect(cnt)
        cv2.rectangle(wbc_excl, (max(0, x-WBC_PAD), max(0, y-WBC_PAD)),
                      (min(w, x+bw+WBC_PAD), min(h, y+bh+WBC_PAD)), 255, -1)

    # Step 2a: RBC counting via Hough Circles
    blur5 = cv2.GaussianBlur(gray, (5, 5), 1)
    circles = cv2.HoughCircles(blur5, cv2.HOUGH_GRADIENT, dp=1,
        minDist=RBC_MIN_DIST, param1=RBC_PARAM1, param2=rbc_param2,
        minRadius=RBC_MIN_R, maxRadius=RBC_MAX_R)
    rbc_centers = []
    if circles is not None:
        for (cx, cy, r) in np.round(circles[0]).astype(int):
            cc = int(np.clip(cx, 0, w-1)); rc = int(np.clip(cy, 0, h-1))
            if wbc_excl[rc, cc] == 0:
                rbc_centers.append((int(cx), int(cy), int(r)))

    # Step 2b: Watershed boundaries
    ws_markers = _hough_seeded_watershed(img_bgr, rbc_centers, h, w)

    # Step 3: PLT
    rbc_excl = np.zeros((h, w), np.uint8)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(rbc_excl, (cx, cy), r+10, 255, -1)
    all_excl  = cv2.bitwise_or(wbc_excl, rbc_excl)
    purp_mask = cv2.inRange(hsv, (PLT_H_LO, PLT_S_MIN, 30), (PLT_H_HI, 255, PLT_V_HI))
    purp_mask = cv2.bitwise_and(purp_mask, cv2.bitwise_not(all_excl))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_OPEN,  np.ones((2, 2), np.uint8))
    purp_mask = cv2.morphologyEx(purp_mask, cv2.MORPH_CLOSE, np.ones((4, 4), np.uint8))
    plt_all, _ = cv2.findContours(purp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plt_dets = [c for c in plt_all if plt_min_area <= cv2.contourArea(c) <= PLT_MAX_AREA]

    # Collect bounding boxes
    wbc_boxes = [[x, y, x+bw, y+bh]
                 for cnt in wbc_cnts for (x, y, bw, bh) in [cv2.boundingRect(cnt)]]
    rbc_boxes = [[max(0, cx-r), max(0, cy-r), min(w, cx+r), min(h, cy+r)]
                 for (cx, cy, r) in rbc_centers]
    plt_boxes = [[x, y, x+bw, y+bh]
                 for cnt in plt_dets for (x, y, bw, bh) in [cv2.boundingRect(cnt)]]

    # Draw result
    result = img_bgr.copy()
    boundary = (ws_markers == -1).astype(np.uint8)
    boundary = cv2.dilate(boundary, np.ones((2, 2), np.uint8), iterations=1)
    result[boundary > 0] = [0, 220, 220]
    for box in wbc_boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(result, (x1, y1), (x2, y2), CLASS_COLORS['WBC'], 2)
        cv2.putText(result, 'WBC', (x1, max(0, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, CLASS_COLORS['WBC'], 1)
    for (cx, cy, r) in rbc_centers:
        cv2.circle(result, (cx, cy), r, CLASS_COLORS['RBC'], 1)
    for box in plt_boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(result, (x1, y1), (x2, y2), CLASS_COLORS['PLT'], 1)

    boxes = {'WBC': wbc_boxes, 'RBC': rbc_boxes, 'PLT': plt_boxes}
    return boxes, result


def draw_gt(img, gt_boxes):
    out = img.copy()
    for name, boxes in gt_boxes.items():
        color = GT_COLORS[name]
        for box in boxes:
            x1, y1, x2, y2 = box
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 1)
            cv2.putText(out, name, (x1, max(0, y1-4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)
    return out


# ─────────────────────── Streamlit UI ────────────────────────────────
def main():
    st.set_page_config(page_title='血液抹片細胞偵測系統', page_icon='🔬', layout='wide')
    st.title('🔬 血液抹片細胞自動辨識與計數系統')
    st.caption('影像處理 114-2 期末專案 ｜ 使用傳統影像處理技術（無深度學習）')

    if DATASET_ROOT is None:
        st.error('找不到 TXL-PBC 資料集！請先 clone 資料集再重啟 App。')
        st.stop()

    st.divider()

    with st.sidebar:
        st.header('⚙️ 設定')
        split     = st.selectbox('資料集分割', ['test', 'val', 'train'], index=0)
        img_files = load_image_list(split)
        if not img_files:
            st.error(f'找不到 {split} 影像。'); st.stop()

        st.subheader('📷 影像選擇')
        img_idx = st.slider(f'影像（共 {len(img_files)} 張）', 0, len(img_files)-1, 0)
        st.caption(Path(img_files[img_idx]).name[:35])

        st.subheader('🎛️ 偵測參數')
        wbc_thresh   = st.slider('WBC 灰階閾值', 90, 160, 130, 5)
        rbc_param2   = st.slider('RBC Hough 靈敏度', 6, 25, 12, 1)
        plt_min_area = st.slider('PLT 最小面積 px²', 100, 1000, 400, 50)

        st.divider()
        st.info('**GT 顏色（細線）**\n\n🟩 WBC　🟡 RBC　🟣 PLT\n\n**預測顏色（粗線）**\n\n🟢 WBC　🔴 RBC　🔵 PLT\n\n🟡 Watershed RBC 分割邊界')

    img_path = img_files[img_idx]
    stem     = Path(img_path).stem
    lbl_path = f'{DATASET_ROOT}/labels/{split}/{stem}.txt'
    img      = cv2.imread(img_path)
    gt       = load_gt(lbl_path)

    with open(img_path, 'rb') as f:
        img_bytes = f.read()

    with st.spinner('偵測中...'):
        boxes, result = detect_cells_cached(
            img_bytes, wbc_thresh, rbc_param2, plt_min_area)

    # Pipeline 視覺化
    st.subheader('📊 影像前處理 Pipeline')
    steps = preprocess_pipeline(img, wbc_thresh)
    c1, c2, c3, c4, c5 = st.columns(5)
    c1.image(bgr2rgb(img),                    caption='① 原始影像',           use_container_width=True)
    c2.image(gray2rgb(steps['grayscale']),     caption='② 灰階',               use_container_width=True)
    c3.image(gray2rgb(steps['gaussian_blur']), caption='③ Gaussian Blur',      use_container_width=True)
    c4.image(gray2rgb(steps['adaptive_thr']),  caption='④ Adaptive Threshold', use_container_width=True)
    c5.image(gray2rgb(steps['morph_close']),   caption='⑤ Morph. Closing',     use_container_width=True)

    st.divider()

    # GT vs 預測
    st.subheader('🎯 偵測結果 vs Ground Truth')
    cgt, cpred = st.columns(2)
    cgt.image(bgr2rgb(draw_gt(img, gt)), caption='Ground Truth', use_container_width=True)
    cpred.image(bgr2rgb(result), caption='預測結果（黃線 = Watershed RBC 分割邊界）', use_container_width=True)

    st.divider()

    # IoU 指標
    st.subheader('📈 偵測指標（IoU ≥ 0.5）')
    cols = st.columns(3)
    for i, name in enumerate(['WBC', 'RBC', 'PLT']):
        gt_b = gt[name]; pred_b = boxes[name]
        tp, fp, fn = match_detections(gt_b, pred_b)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        with cols[i]:
            st.metric(name, f'GT:{len(gt_b)} / Pred:{len(pred_b)}')
            st.write(f'TP={tp}  FP={fp}  FN={fn}')
            st.write(f'Precision: **{prec:.1%}**')
            st.write(f'Recall: **{rec:.1%}**')

    st.divider()

    # 下載
    st.subheader('💾 下載結果')
    cd1, cd2 = st.columns(2)
    _, r_enc = cv2.imencode('.png', result)
    cd1.download_button('⬇️ 偵測結果影像', r_enc.tobytes(), f'detection_{stem}.png', 'image/png')
    _, g_enc = cv2.imencode('.png', draw_gt(img, gt))
    cd2.download_button('⬇️ Ground Truth 影像', g_enc.tobytes(), f'groundtruth_{stem}.png', 'image/png')


if __name__ == '__main__':
    main()

## Cell 11 — 啟動 Streamlit App（Colab 環境，cloudflared 公開 URL）

執行後會自動輸出一個 `*.trycloudflare.com` 公開網址，點擊即可在任何瀏覽器開啟。

> **VSCode 連接 Colab 用戶**：也可改用 Cell 12（透過 VSCode Port Forwarding，不需公開 URL）

In [ ]:
import subprocess, time, re, os

CF_URL  = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
CF_BIN  = './cloudflared'

# ── 下載 cloudflared（wget 失敗自動改用 curl）─────────────────────
if not os.path.exists(CF_BIN):
    print('下載 cloudflared...')
    ok = subprocess.run(['wget', '-q', CF_URL, '-O', CF_BIN]).returncode == 0
    if not ok:
        ok = subprocess.run(['curl', '-sL', CF_URL, '-o', CF_BIN]).returncode == 0
    if not ok:
        raise RuntimeError('cloudflared 下載失敗，請檢查網路或改用 Cell 12（VSCode Port Forwarding）')
    subprocess.run(['chmod', '+x', CF_BIN])
    print('下載完成')

# ── 啟動 Streamlit（背景）──────────────────────────────────────────
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port=8501',
     '--server.headless=true',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

# ── 啟動 cloudflared tunnel，從 stderr 讀取公開 URL ───────────────
cf_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE
)

url = None
for line in cf_proc.stderr:
    text = line.decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', text)
    if match:
        url = match.group(0)
        break

print('Streamlit App 已啟動')
print(f'公開網址：{url}')
print('點擊連結即可開啟互動式 App')